# Transfer Additional US event to ds HUC10 Model 

#Goal 

Identify the us events that are driving the recurrence interval values at the tie-in and transfer those dss files to the downstream HUC, create new flow files and plans and then update the prj file to include the new information. 

In [1]:
import os
import re
import pathlib as pl

In [2]:
os.chdir('..')
home = pl.Path(os.getcwd())

#user to set variables for the project. tagged as parameter for papermill runs
project = 'wy_fy22'

#target huc
target = '1404010608'

In [3]:
print('home is at: ',home)
home = pl.Path(home)
from src.hdf import *

inputs = home/'inputs'
outputs_base = home/'outputs'

export_folder = outputs_base/project/'trial_us_to_ds_events'
target_export = export_folder/str('wy_gdg_'+target)
assert home.stem == '_code', 'restart kernel and rerun code'

home is at:  U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code


In [4]:
#location of upstream HUC dss files for connections between different HUC8s
transfer_dss_location = inputs/project/'transfer_dss'

In [5]:
#open dictionaries to understand which HUC the upstream one flows into and the downstream junction for each
with open(inputs/project/'dictionaries'/'HUC10_outflow_toHUC10.json') as src:
    huc_connect_huc = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_into_dsJunction.json') as src:
    huc_connect_j = json.load(src)
with open(inputs/project/'dictionaries'/'junc_res_sink_next_junc_down.json') as src:
    j_to_j = json.load(src)
with open(inputs/project/'dictionaries'/'HUC10_Junctions.json') as src:
    huc_js = json.load(src)
with open(inputs/project/'dictionaries'/'source_if_reservoir.json') as src:
    src_js = json.load(src)
with open(inputs/project/'dictionaries'/'hms_feature_origin_dict.json') as src:
    src_huc_hms = json.load(src)
with open(inputs/project/'dictionaries'/'outofscope_HUC10_to_HUC10.json') as src:
    out_of_scope_huc_inflows = json.load(src)
with open(inputs/project/'dictionaries'/'Junction_Subbasins.json') as src:
    j_connect_sub = json.load(src)

In [6]:
#create folder for huc outputs
if not os.path.exists(target_export):
    os.makedirs(target_export)
    
#creating interior hydrology folder 
if not os.path.exists(target_export/'Hydrology'):
    os.mkdir(target_export/'Hydrology')

In [7]:
#this is the us huc desired to be selected. would attempt a list but a lot of pre-written code works for singular HUC
us_hucs = []
ds_j = []

#create HUC BC connections
bc_connections = {'junctions':{},'dss_path':{},'ts':{}}

for key, val in huc_connect_huc.items():
    if val == target:
        us_hucs.append(key)
        ds_j.append(huc_connect_j[key])
        bc_connections['junctions'][key] = huc_connect_j[key]
        bc_connections['dss_path'][key] = {}

In [8]:
bc_connections


{'junctions': {'1404010603': 'HUC_106_J_91',
  '1404010604': 'HUC_106_J_93',
  '1404010605': 'HUC_106_J_229'},
 'dss_path': {'1404010603': {}, '1404010604': {}, '1404010605': {}},
 'ts': {}}

In [9]:
ds_junc_adjustments = {'1404010109':'HUC_101_J_277',
                       '1404010401':'HUC_104_J_307',
                       '1404010602':'HUC_106_J_42',
                       '1404010903':'HUC_109_J_156',
                       '1404010705':'HUC_107_J_220'} 

In [10]:
#need to update bc_connections dictionary if the junction cannot be located in the html/geojson files. 
#if it needs adjustments, it will be in the ds_junc_adjustments dictionary
original_acting_junction = {}
for huc in us_hucs:
    if huc in ds_junc_adjustments.keys():
        print(f"Warning: temporary ds_junction adjustment required for {huc}")
        original_acting_junction[huc] = bc_connections['junctions'][huc]
        
        #temporarily update the bc_connection dictionary to ensure the correct events are selected
        bc_connections['junctions'][huc] = ds_junc_adjustments[huc]
    
    else:
        pass
        
        
print("original junction has been saved in separate dictionary for future use:",original_acting_junction)

original junction has been saved in separate dictionary for future use: {}


In [11]:
bc_connections

{'junctions': {'1404010603': 'HUC_106_J_91',
  '1404010604': 'HUC_106_J_93',
  '1404010605': 'HUC_106_J_229'},
 'dss_path': {'1404010603': {}, '1404010604': {}, '1404010605': {}},
 'ts': {}}

In [12]:
events_dict_us = {}
#identify the controlling upstreamevent at the tie-in
for huc, j in  bc_connections['junctions'].items():
    events_dict_us[huc] = {j:{}}
    htmls = glob.glob(str(inputs/project/huc[4:8]/f'HUC{huc[:8]}'/huc/'plots'/'*_map.html'))
    events_dict_us = get_sst_storms_by_recurrence_us_huc(events_dict_us,huc,htmls,j)

In [13]:
events_dict_us

{'1404010603': {'HUC_106_J_91': {'0.002': 'R8-Y408-E0001',
   '0.01m': 'R6-Y390-E0001',
   '0.01p': 'R7-Y261-E0001',
   '0.01': 'R3-Y482-E0006',
   '0.02': 'R9-Y546-E0003',
   '0.04': 'R2-Y104-E0003',
   '0.1': 'R6-Y304-E0005'}},
 '1404010604': {'HUC_106_J_93': {'0.002': 'R8-Y164-E0002',
   '0.01m': 'R6-Y108-E0001',
   '0.01p': 'R3-Y377-E0002',
   '0.01': 'R1-Y402-E0001',
   '0.02': 'R7-Y058-E0007',
   '0.04': 'R9-Y021-E0002',
   '0.1': 'R9-Y359-E0001'}},
 '1404010605': {'HUC_106_J_229': {'0.002': 'R5-Y077-E0002',
   '0.01m': 'R6-Y171-E0002',
   '0.01p': 'R7-Y212-E0008',
   '0.01': 'R7-Y012-E0003',
   '0.02': 'R9-Y479-E0003',
   '0.04': 'R1-Y107-E0005',
   '0.1': 'R6-Y164-E0003'}}}

In [14]:
#Create dictionary with all the events required at to the upstream HUCs
complete_us_dict = {}

for huc in events_dict_us.keys():
    prob_shps = glob.glob(str(inputs/project/huc[4:8]/f'HUC{huc[:8]}'/huc/'*.geojson'))
    complete_us_dict.update(get_sst_storms_by_recurrence(huc,prob_shps))

Note that the same storm event R5-Y077-E0002 is used for more than recurrence interval


In [15]:
complete_us_dict

{'1404010603': {'0.002': ['R6-Y164-E0001', 'R3-Y278-E0002', 'R8-Y408-E0001'],
  '0.01m': ['R1-Y535-E0001', 'R6-Y390-E0001'],
  '0.01p': ['R8-Y053-E0003', 'R2-Y497-E0001', 'R7-Y261-E0001'],
  '0.01': ['R6-Y511-E0001', 'R2-Y497-E0001', 'R3-Y482-E0006'],
  '0.02': ['R9-Y160-E0005', 'R9-Y546-E0003'],
  '0.04': ['R8-Y100-E0002', 'R2-Y104-E0003'],
  '0.1': ['R4-Y358-E0003', 'R6-Y304-E0005']},
 '1404010604': {'0.002': ['R3-Y127-E0005', 'R9-Y092-E0003', 'R8-Y164-E0002'],
  '0.01m': ['R4-Y170-E0002', 'R6-Y108-E0001'],
  '0.01p': ['R3-Y236-E0001', 'R3-Y377-E0002'],
  '0.01': ['R3-Y236-E0001', 'R1-Y402-E0001'],
  '0.02': ['R7-Y146-E0003', 'R7-Y058-E0007'],
  '0.04': ['R9-Y302-E0002', 'R9-Y021-E0002'],
  '0.1': ['R6-Y388-E0001', 'R9-Y359-E0001']},
 '1404010605': {'0.002': ['R5-Y077-E0002'],
  '0.01m': ['R6-Y171-E0002'],
  '0.01p': ['R7-Y212-E0008'],
  '0.01': ['R7-Y012-E0003'],
  '0.02': ['R1-Y262-E0002', 'R9-Y479-E0003'],
  '0.04': ['R1-Y485-E0005', 'R1-Y107-E0005'],
  '0.1': ['R6-Y164-E0003']}}

In [16]:
#Now that the events have been found, the following dictionaries must be fixed with the original junction value.
#dictionaries that require fixing

for huc in original_acting_junction:
    bc_connections['junctions'][huc] = original_acting_junction[huc]
    
    temp_save = events_dict_us[huc][ds_junc_adjustments[huc]]
    del events_dict_us[huc][ds_junc_adjustments[huc]]
    events_dict_us[huc][original_acting_junction[huc]] = temp_save

In [17]:
bc_connections

{'junctions': {'1404010603': 'HUC_106_J_91',
  '1404010604': 'HUC_106_J_93',
  '1404010605': 'HUC_106_J_229'},
 'dss_path': {'1404010603': {}, '1404010604': {}, '1404010605': {}},
 'ts': {}}

In [18]:
events_dict_us

{'1404010603': {'HUC_106_J_91': {'0.002': 'R8-Y408-E0001',
   '0.01m': 'R6-Y390-E0001',
   '0.01p': 'R7-Y261-E0001',
   '0.01': 'R3-Y482-E0006',
   '0.02': 'R9-Y546-E0003',
   '0.04': 'R2-Y104-E0003',
   '0.1': 'R6-Y304-E0005'}},
 '1404010604': {'HUC_106_J_93': {'0.002': 'R8-Y164-E0002',
   '0.01m': 'R6-Y108-E0001',
   '0.01p': 'R3-Y377-E0002',
   '0.01': 'R1-Y402-E0001',
   '0.02': 'R7-Y058-E0007',
   '0.04': 'R9-Y021-E0002',
   '0.1': 'R9-Y359-E0001'}},
 '1404010605': {'HUC_106_J_229': {'0.002': 'R5-Y077-E0002',
   '0.01m': 'R6-Y171-E0002',
   '0.01p': 'R7-Y212-E0008',
   '0.01': 'R7-Y012-E0003',
   '0.02': 'R9-Y479-E0003',
   '0.04': 'R1-Y107-E0005',
   '0.1': 'R6-Y164-E0003'}}}

In [19]:
#attain a list of dss files already exported during the autobc process for the target huc
dss_basename_ds = []

dss_files_ds = glob.glob(str(outputs_base/project/f'wy_gdg_{target}'/'[Hh]ydrology'/'*.dss'))
for path in dss_files_ds:
    path_base = os.path.basename(path).replace('_output.dss','')
    dss_basename_ds.append(path_base.replace('_','-'))
    
print(dss_basename_ds, len(dss_basename_ds))

['R10-Y055-E0001', 'R1-Y423-E0001', 'R1-Y468-E0005', 'R1-Y476-E0001', 'R1-Y509-E0002', 'R2-Y421-E0002', 'R2-Y474-E0001', 'R3-Y050-E0001', 'R5-Y274-E0001', 'R6-Y181-E0005', 'R6-Y255-E0001', 'R6-Y482-E0002', 'R7-Y162-E0003', 'R8-Y089-E0001', 'R8-Y475-E0004', 'R9-Y252-E0003', 'R9-Y461-E0001'] 17


In [20]:
#create a dictionary with available dss files from the autobc output for each of the upstream HUCs
huc_dss_base = {}
#the paths for all of the dss files available for all upstream HUCs
dss_files_tot = set()

for huc in events_dict_us.keys():
    dss_files = glob.glob(str(outputs_base/project/f'wy_gdg_{huc}'/'[Hh]ydrology'/'*.dss'))
    dss_basenames = []
    for path in dss_files:
        dss_files_tot.add(path)
        dss_basenames.append(os.path.basename(path).replace('_output.dss',''))
    huc_dss_base.update({huc: dss_basenames})
    print(len(dss_files),f': amount of dss files identified for huc {huc}')




16 : amount of dss files identified for huc 1404010603
14 : amount of dss files identified for huc 1404010604
9 : amount of dss files identified for huc 1404010605


In [21]:
len(dss_files_tot)

39

In [22]:
#adding events from us dictionary if not already in the downstream model export
events_req = set()
for huc in events_dict_us.keys():
    for junc in events_dict_us[huc].keys():
        for rec, event in events_dict_us[huc][junc].items():
            if event not in dss_basename_ds:
                events_req.add(event)
            else:
                print(f'event identified already in target huc {target} Hydrology folder:', event)
                pass
print("Required events =",events_req, len(events_req))

Required events = {'R8-Y164-E0002', 'R8-Y408-E0001', 'R9-Y359-E0001', 'R6-Y171-E0002', 'R6-Y108-E0001', 'R9-Y546-E0003', 'R5-Y077-E0002', 'R7-Y212-E0008', 'R3-Y377-E0002', 'R6-Y164-E0003', 'R7-Y261-E0001', 'R7-Y012-E0003', 'R1-Y402-E0001', 'R3-Y482-E0006', 'R1-Y107-E0005', 'R7-Y058-E0007', 'R6-Y304-E0005', 'R9-Y021-E0002', 'R2-Y104-E0003', 'R9-Y479-E0003', 'R6-Y390-E0001'} 21


In [23]:
events_req

{'R1-Y107-E0005',
 'R1-Y402-E0001',
 'R2-Y104-E0003',
 'R3-Y377-E0002',
 'R3-Y482-E0006',
 'R5-Y077-E0002',
 'R6-Y108-E0001',
 'R6-Y164-E0003',
 'R6-Y171-E0002',
 'R6-Y304-E0005',
 'R6-Y390-E0001',
 'R7-Y012-E0003',
 'R7-Y058-E0007',
 'R7-Y212-E0008',
 'R7-Y261-E0001',
 'R8-Y164-E0002',
 'R8-Y408-E0001',
 'R9-Y021-E0002',
 'R9-Y359-E0001',
 'R9-Y479-E0003',
 'R9-Y546-E0003'}

### Copying over dss files not already in downstream model

In [24]:
dss_files_model = []
for event in events_req:
    evnt_name = re.search('Y\d+-E\d+',event).group()
    evnt_name_u = evnt_name.replace('-','_')
    criteria = f'\S+(R\d+-)?{evnt_name_u}_output.dss'
    r = re.compile(criteria)
    dss_matches = list(filter(r.match, dss_files_tot))
    # print(dss_matches)
    if len(dss_matches) > 1:
        evnt_name_full = re.search('R\d+-Y\d+-E\d+',event).group()
        evnt_name_full_u = evnt_name_full.replace('-','_')
        criteria = f'\S+{evnt_name_full_u}_output.dss' 
        r = re.compile(criteria)
        dss_matches = list(filter(r.match, dss_matches))
    assert len(dss_matches) != 0, "there are no dss files matches"
    # assert len(dss_matches) == 1, "there are more than 1 dss files with the same Y and E number that don't have a R number specified"
    out_name = event.replace('-','_')+"_output.dss"
    
    #quick check to see if the event being copied over is from the same HUC8 as our target. This is a precaution to ensure the junction information will be included in the unsteady flow information copied over 
    if f'wy_gdg_{target[:8]}' not in dss_matches[0]:
        print('dss file is being copied from a HUC10 in a different HUC8: Event ', event)
        from_huc10_base = re.search(r'\\wy_gdg_1404......\\', dss_matches[0])
        from_huc8 = str(from_huc10_base.group())[8:-3]
        
        event_temp = event.replace('-','_')
        # event_mod = event_[:1]+'-'+event_[1:]
        print(f'attempting to pull dss file {event_temp} from {from_huc8}')
        
        shutil.copy(transfer_dss_location/target[:8]/f'{event_temp}_output.dss',target_export/'Hydrology'/out_name)
        dss_files_model.append(str(target_export/'Hydrology'/out_name))
    else:
        print(f'copying existing dss file {dss_matches[0]} for event {event}from auto-bc creation folder')
        shutil.copy(dss_matches[0],target_export/'Hydrology'/out_name)
        dss_files_model.append(str(target_export/'Hydrology'/out_name))

copying existing dss file U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010604\Hydrology\R8_Y164_E0002_output.dss for event R8-Y164-E0002from auto-bc creation folder
copying existing dss file U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010603\Hydrology\R8_Y408_E0001_output.dss for event R8-Y408-E0001from auto-bc creation folder
copying existing dss file U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gdg_1404010604\Hydrology\R9_Y359_E0001_output.dss for event R9-Y359-E0001from auto-bc creation folder
copying existing dss file U:\173432208011\studies\1_great_divide_green_watershed\production\engineering_riverine\hydraulics\hydrology_incorporation\_code\outputs\wy_fy22\wy_gd

In [25]:
dss_files_model

['U:\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010608\\Hydrology\\R8_Y164_E0002_output.dss',
 'U:\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010608\\Hydrology\\R8_Y408_E0001_output.dss',
 'U:\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010608\\Hydrology\\R9_Y359_E0001_output.dss',
 'U:\\173432208011\\studies\\1_great_divide_green_watershed\\production\\engineering_riverine\\hydraulics\\hydrology_incorporation\\_code\\outputs\\wy_fy22\\trial_us_to_ds_events\\wy_gdg_1404010608\\Hydrology\\R6_Y171_E0002_output.dss',
 'U:\\173432208011\\studies\\1_great_divide_green_wa

In [26]:
#copy over the project file for the target huc from the autobc export 
if not os.path.exists(target_export/f'wy_gdg_{target}.prj'):
        prj_out_name = shutil.copy(outputs_base/project/f'wy_gdg_{target}'/f'wy_gdg_{target}.prj',target_export/f'wy_gdg_{target}.prj')
else:
        prj_out_name = target_export/f'wy_gdg_{target}.prj'

### Copying over Geometry files

In [27]:
model_name = f'wy_gdg_{target}'

geo_out_hdf = shutil.copy(outputs_base/project/model_name/f'{model_name}.g51.hdf',target_export/f'{model_name}.g51.hdf')
geo_out = shutil.copy(outputs_base/project/model_name/f'{model_name}.g51',target_export/f'{model_name}.g51')

In [28]:
hf_geo = h5py.File(str(geo_out_hdf),'r')

#domain names
domain_geo = str(list(hf_geo['Geometry']['2D Flow Areas']['Attributes'])[0][0]).strip("'b\'")

In [29]:
plan_path, in_geom_path, in_flow_path = get_current_ras_files(outputs_base/project/model_name/f'{model_name}.prj')

## Complete Dictionary with included events as desired - May be exported for one time completion 

#list of all applicable HUC10s
huc_connect_huc.keys()

complete_dictionary_p1 = {}

for huc in huc_connect_huc.keys():
    prob_shps = glob.glob(str(inputs/project/huc[4:8]/f'HUC{huc[:8]}'/huc/'*.geojson'))
    complete_dictionary_p1.update(get_sst_storms_by_recurrence(huc,prob_shps))

#manual adjustments required for some of the HUC models (ds_junc does not fall within the plot and thus it cannot find the events related to it. 
#The dictionary requires that the upstream junction be used. select one in similar stream as the ds_junc)

ds_junc_adjustments = {'1404010109':'HUC_101_J_277',
                       '1404010401':'HUC_104_J_307',
                       '1404010602':'HUC_106_J_42',
                       '1404010903':'HUC_109_J_156',
                       '1404010705':'HUC_107_J_220'} 



#create separate temporary dictionary to add to the list shown in the events
completed_us_dictionary_p2 = {}

for huc in huc_connect_huc.keys():
    us_huc_temp = [k for k,v in huc_connect_huc.items() if v == huc]
    if len(us_huc_temp) == 0:
        print(f'huc {huc} is most upstream')
    else:
        bc_connections = {'junctions':{},'dss_path':{},'ts':{}}
        
        us_hucs = [k for k in us_huc_temp]
        ds_j = [huc_connect_j[k] for k in us_huc_temp]
        
        for us in us_hucs:
            if us not in ds_junc_adjustments.keys():
                bc_connections['junctions'][us] = huc_connect_j[us]
                bc_connections['dss_path'][us] = {}
            else:
                bc_connections['junctions'][us] = ds_junc_adjustments[us]
                bc_connections['dss_path'][us] = {}
                
        events_dict_us = {}
        print(f'filling out dictionary for huc {huc}')
        for huc, j in  bc_connections['junctions'].items():
            events_dict_us[huc] = {j:{}}
            htmls = glob.glob(str(inputs/project/huc[4:8]/f'HUC{huc[:8]}'/huc/'plots'/'*_map.html'))
            events_dict_us = get_sst_storms_by_recurrence_us_huc(events_dict_us,huc,htmls,j)
            
        completed_us_dictionary_p2.update(events_dict_us)

len(completed_us_dictionary_p2)

complete_dictionary_p1

for key in completed_us_dictionary_p2:
    ds_huc = huc_connect_huc[key]
    # print(f'huc {key} goes to huc {ds_huc}')
    for rec in completed_us_dictionary_p2[key].values():
        for i,j in rec.items():
            event_list = complete_dictionary_p1[ds_huc][i]
            print(ds_huc, i, event_list, j)
            
            if j not in event_list:
                event_list.append(j)
                complete_dictionary_p1[ds_huc][i] = event_list
            else:
                pass

import json
#export master combined list to json. should only be done once cannot be redone or may overwrite previous list.
with open(export_folder/'completed_event_dictionary.json', "w") as outfile:
    json.dump(complete_dictionary_p1, outfile, indent= 1)

#WARNING --- This dictionary has had the downstream junctions modified for a select number of HUCs in order to attain the events related to it. Be advised they will not match ds junctions 100%

import json
#export master combined list to json. should only be done once cannot be redone or may overwrite previous list.
with open(export_folder/'tie_in_dictionary.json', "w") as outfile2:
    json.dump(completed_us_dictionary_p2, outfile2, indent= 1)

## Update the files from AutoBC creation output 

In [30]:
## Once we have the new plans for tie-ins we need to add them to 
# the existing plan and flow files and then update the prj file

#get max plan value as start_id
start_id = 51
plan_files = glob.glob(str(plan_path.parent/'*.p*[!rj][!.hdf]'))
for p in plan_files:
    p_num = re.search('.p[\d]+',p)
    if int(p_num.group()[2:]) >= start_id:
        start_id += 1

In [31]:
start_id

68

### Add in additional dss information per auto BC creation workflow

In [32]:
outputs = outputs_base/project/model_name

In [33]:
## ext bc line info
if os.path.exists(f'{outputs}/{target}_ext_forcing_bc_lines.shp'):
    ext_bcs = gpd.read_file(f'{outputs}/{target}_ext_forcing_bc_lines.shp')
else:
    ext_bcs = pd.DataFrame(columns=['empty'])

ext_bc_dict = ext_bcs.to_dict()
ext_index = ext_bcs.index.tolist()
ext_bc_dict['in_flow_path'] = in_flow_path
ext_bc_dict['in_plan_path'] = plan_path

In [34]:
## ext bc oos line info
if os.path.exists(f'{outputs}/{target}_ext_forcing_bc_lines_oos.shp'):
    ext_bcs_oos = gpd.read_file(f'{outputs}/{target}_ext_forcing_bc_lines_oos.shp')
    ext_bcs_combined = pd.concat([ext_bcs, ext_bcs_oos])
    ext_bcs_combined.reset_index(inplace=True,drop=True)
    ext_bc_dict_combined = ext_bcs_combined.to_dict()
    ext_index_combined = ext_bcs_combined.index.tolist()
else:
    ext_bc_dict_oos = {}
    ext_index_oos = []
    ext_bc_dict_combined = ext_bc_dict
    ext_index_combined = ext_index


ext_bc_dict_combined['in_flow_path'] = in_flow_path
ext_bc_dict_combined['in_plan_path'] = plan_path

In [35]:
## int bc line info
if os.path.exists(f'{outputs}/{target}_in_forcing_bc_lines.shp'):
    int_bcs = gpd.read_file(f'{outputs}/{target}_in_forcing_bc_lines.shp')
else:
    int_bcs = pd.DataFrame(columns=['empty'])

int_bc_dict = int_bcs.to_dict()
int_index = int_bcs.index.tolist()
int_bc_dict['in_flow_path'] = in_flow_path
int_bc_dict['in_plan_path'] = plan_path

In [36]:
## flow reduction reaches
# if os.path.exists(f'{outputs}/{target}_flow_reduction_forcing_bc_lines.shp'):
#     reaches_r_s = gpd.read_file(f'{outputs}/{target}_flow_reduction_forcing_bc_lines.shp')
# else:
#     reaches_r_s = pd.DataFrame(columns=['empty'])
    
# int_r_bc_dict = reaches_r_s.to_dict()
# int_r_index = reaches_r_s.index.tolist()
# int_r_bc_dict['in_flow_path'] = in_flow_path
# int_r_bc_dict['in_plan_path'] = plan_path
#get hms schematic information

schem_folder = inputs/project/target[4:8]/f'HUC{target[:8]}_SST_schematic'
if os.path.exists(schem_folder/'hms_plotting'):
    schem_folder = schem_folder/'hms_plotting'
# junctions = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Junction.shp'))
# sinks = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Sink.shp'))
# if os.path.exists(str(schem_folder/f'HUC{huc[:8]}_SST_Reservoir.shp')):
#     reservoirs = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Reservoir.shp'))
#     hms_points = pd.concat([junctions, sinks,reservoirs]).set_index('index')
# else:
#     hms_points = pd.concat([junctions, sinks]).set_index('index')
# subbasins = gpd.read_file(str(schem_folder/f'HUC{huc[:8]}_SST_Subbasin.shp'))
reaches = gpd.read_file(str(schem_folder/f'HUC{target[:8]}_SST_Reach.shp'))




#get reach junction dictionary
with open(inputs/project/'dictionaries'/'reachfromjunction.json') as src:
    reach_to_us_junction = json.load(src)
##filter to relevant flow reduction reaches\
reaches_col = reaches.columns.to_list()
if 'Channel Lo' in reaches_col:
    cl = 'Channel Lo'
    sb = 'Channel _1'
    rd = 'Channel _2'
else:
    cl = 'Channel _1'
    sb = 'Channel _2'
    rd = 'Channel _3'
flow_r_filter = reaches[cl] == 'Constant'
#channel 2 is initial flow reduction, channel 3 is % reduction of flow hydrograph so adjusted flow hydrograph = (inflow - 5) *.8
reaches_r = reaches.loc[flow_r_filter]
#filter by relevant huc
if target[:8] == '14040102':
    ds = 'downstream'
else:
    ds = 'Downstream'
reaches_r_s = reaches_r.copy().loc[reaches_r[ds].isin(huc_js[target]) == True]
if not reaches_r_s.empty:
    reaches_r_s['subtraction'] = reaches_r_s[sb]
    reaches_r_s['reduction'] = reaches_r_s[rd]
else:
    reaches_r_s['subtraction'] = 0
    reaches_r_s['reduction'] = 0
    
reaches_r_s['upstream'] = reaches_r_s['name'].apply(lambda x: reach_to_us_junction[x])

####modify to make useful for bc purposes###
#to be consistent with ext code
reaches_r_s['junction'] = reaches_r_s.name
#Internal BC EG slope should only impact the normal depth calc used to distribute flow to adjacent cells. 
#Assume uniform value of 0.01 (relatively steep) to force flow to the low point of the cell
reaches_r_s['slope'] = 0.01

reaches_r_s.reset_index(inplace=True)

int_r_bc_dict = reaches_r_s.to_dict()
int_r_index = reaches_r_s.index.tolist()
int_r_bc_dict['in_flow_path'] = in_flow_path
int_r_bc_dict['in_plan_path'] = plan_path

In [37]:
## flow sources
if os.path.exists(f'{outputs}/{target}_in_forcing_bc_lines_sources.shp'):
    src_bcs = gpd.read_file(f'{outputs}/{target}_in_forcing_bc_lines_sources.shp')
else:
    src_bcs = pd.DataFrame(columns=['empty'])
src_bc_dict = src_bcs.to_dict()
src_index = src_bcs.index.tolist()
src_bc_dict['in_flow_path'] = in_flow_path
src_bc_dict['in_plan_path'] = plan_path

######################### Latest Additions

In [38]:
#plans and unsteady flow files are not recognized over 100 by the models in naming. Add clause to resolve if added dss_files are over

if len(dss_files_model)+start_id <= 100:
    dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model,ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,start_id,src_huc_hms,append=True)

else:
    fixed_id_temp = 100 - start_id
    dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model[0:fixed_id_temp],ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,start_id,src_huc_hms,append=True)
    
    new_start_id = 51 - (len(dss_files_model)-fixed_id_temp)
    dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model[fixed_id_temp:],ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,new_start_id,src_huc_hms,append=True)

print('\nCompleted')


Completed


#########################

############################# write dss files to run events using subbasin inflows = original draft
dss_matches = write_updated_ext_bc_files(domain_geo,dss_files_model,ext_index_combined,ext_bc_dict_combined,int_index,int_bc_dict,int_r_index,int_r_bc_dict,src_index, src_bc_dict, inputs,target_export,project,target,start_id,src_huc_hms,append=True)

In [39]:
#Complete

In [40]:
print('complete')

complete
